In [1]:
import numpy as np
import pandas as pd
import glob # glob is a built-in module used to search for files and directories on your system that match a specified pattern.

In [2]:
import os
os.getcwd()

'/home/blackbird/Documents/eda-capstone-project/notebook'

Verifying the files before combining, detecting shape, columns name befrore combining files.

In [ ]:
files=glob.glob("../data/raw/*.csv")
# print(files)

for file in files:
    df=pd.read_csv(file)
    print(f"\n File Name: {file}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")


 File Name: ../data/raw/monthly_emi_track.csv
Shape: (2000000, 23)
Columns: ['loan_id', 'installment_due_inr', 'total_emi_due_inr', 'total_emi_paid_inr', 'emi_overdue_inr', 'emi_bounce_count', 'consecutive_missed_emis', 'last_emi_payment_date', 'emi_advance_paid_inr', 'prepayment_flag', 'avg_payment_delay_days', 'payment_mode', 'emi_bank_name', 'penal_charges_inr', 'waiver_granted_flag', 'waiver_amount_inr', 'loan_restructured_flag', 'ots_offered_flag', 'ots_accepted_flag', 'emi_coverage_ratio', 'pdc_count', 'emi_to_income_ratio', 'payment_discipline']

 File Name: ../data/raw/loan_enquiry_bureau.csv
Shape: (2000000, 24)
Columns: ['loan_id', 'num_enquiries_30d', 'num_enquiries_90d', 'num_enquiries_6m', 'num_enquiries_12m', 'num_enquiries_24m', 'primary_enq_purpose', 'unique_lenders_enquired', 'rejected_applications', 'approved_applications', 'rejection_rate_pct', 'income_doc_type', 'kyc_status', 'field_verification_status', 'credit_committee_flag', 'loan_approved_date', 'processing_da

In [ ]:
import gc

In [ ]:
files=glob.glob("../data/raw/*.csv")
for file in files:
    print(file)

../data/raw/monthly_emi_track.csv
../data/raw/loan_enquiry_bureau.csv
../data/raw/payment_history.csv
../data/raw/customer_bureau.csv
../data/raw/credit_card_behavior.csv
../data/raw/branch_region_economy.csv
../data/raw/loans_master.csv
../data/raw/collateral_assets.csv
../data/raw/loan_performance.csv


In [ ]:
files=glob.glob("../data/raw/*.csv")
CHUNK=100_000 # initializing chunk size , safe for RAM 
os.makedirs("../data/processed",exist_ok=True)

def downcast_df(df):
    for col in df.select_dtypes(include=["int64","int32"]).columns:
        df[col]=pd.to_numeric(df[col],downcast="integer")
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col]=pd.to_numeric(df[col],downcast="float")
    for col in df.select_dtypes(include=["object"]).columns:
        if df[col].nunique() / len(df) < 0.05: # less unique values -> category
            df[col] = df[col].astype("category")
    return df

report = []
 
for file in files:
    # print(file)
    key=os.path.basename(file).replace(".csv","")
    chunks=[]
    for chunk in pd.read_csv(file,chunksize=CHUNK): 
        # chunk = downcast_df(chunk) # downcasting while chunking
        chunks.append(chunk)

    df=pd.concat(chunks,ignore_index=True)
    del chunks
    gc.collect() # free chunk list from RAM immediately

    # memory before downcast:
    mem_before=df.memory_usage(deep=True).sum() / 1e6
    
    # downcasting
    df=downcast_df(df)

    # memory after downcast:
    mem_after = df.memory_usage(deep=True).sum() / 1e6

    out_path=f"../data/processed/{key}.parquet"
    df.to_parquet(out_path,index=False,engine="pyarrow",compression="snappy")
    parquet_mb=os.path.getsize(out_path) / 1e6

    report.append({
        "file"      : key,
        "rows"      : df.shape[0],
        "cols"      : df.shape[1],
        "mem_before_mb": round(mem_before,2),
        "mem_after_mb":round(mem_after,2),
        "reduction_pct":round((1-mem_after/mem_before)*100,1),
        "parquet_mb":round(parquet_mb,2)
    })

    print(f"{key:35s} | Before: {mem_before:.1f}MB " f"| After: {mem_after:.1f}MB | Parquet: {parquet_mb:.1f}MB")

report_df=pd.DataFrame(report)
report_df.to_markdown("../report/figures/data_acqu_join_clean.md",index=False)

monthly_emi_track                   | Before: 847.4MB | After: 288.0MB | Parquet: 78.6MB
loan_enquiry_bureau                 | Before: 1163.1MB | After: 192.0MB | Parquet: 42.9MB
payment_history                     | Before: 392.0MB | After: 314.0MB | Parquet: 76.4MB
customer_bureau                     | Before: 1063.4MB | After: 444.0MB | Parquet: 107.9MB
credit_card_behavior                | Before: 615.7MB | After: 200.0MB | Parquet: 47.1MB
branch_region_economy               | Before: 696.4MB | After: 324.0MB | Parquet: 47.6MB
loans_master                        | Before: 1640.7MB | After: 250.0MB | Parquet: 67.3MB
collateral_assets                   | Before: 885.5MB | After: 214.0MB | Parquet: 32.2MB
loan_performance                    | Before: 585.4MB | After: 204.0MB | Parquet: 25.5MB


In [ ]:
report_df.mem_before_mb.sum()

np.float64(7889.61)

In [ ]:
report_df.mem_after_mb.sum()


np.float64(2430.12)

In [ ]:
report_df.parquet_mb.sum()

np.float64(525.48)